# Final Project — Step 1: Bronze Ingestion
## Source: NYC TLC public dataset | Target: local Bronze Delta

- Downloads Yellow Taxi January 2024 Parquet from NYC TLC public source
- Adds ingestion metadata columns (`_ingestion_timestamp`, `_source_file`, etc.)
- Writes to Delta format partitioned by `_ingestion_month`
- No transformations — raw data preserved as-is (Bronze layer principle)

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("final-project-bronze")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print("✓ Spark session created with Delta support")

Spark version: 3.5.0
✓ Spark session created with Delta support


In [2]:
# Paths and constants
BASE_PATH           = "/workspace/output/final_project"
BRONZE_YELLOW_PATH  = f"{BASE_PATH}/bronze/yellow"

# NYC TLC public source — Yellow Taxi, January 2024
SOURCE_URL      = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
SOURCE_FILENAME = "yellow_tripdata_2024-01.parquet"
INGESTION_MONTH = "2024-01"

print(f"Bronze target : {BRONZE_YELLOW_PATH}")
print(f"Source URL    : {SOURCE_URL}")

Bronze target : /workspace/output/final_project/bronze/yellow
Source URL    : https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet


In [3]:
# Download source file to driver local filesystem

import urllib.request
import os

LOCAL_TMP  = "/tmp/nyc_taxi"
os.makedirs(LOCAL_TMP, exist_ok=True)
local_file = f"{LOCAL_TMP}/{SOURCE_FILENAME}"

print(f"Downloading {SOURCE_URL} ...")
urllib.request.urlretrieve(SOURCE_URL, local_file)

size_mb = os.path.getsize(local_file) / (1024 * 1024)
print(f"Downloaded to {local_file} — size: {size_mb:.2f} MB")

Downloaded to /tmp/nyc_taxi/yellow_tripdata_2024-01.parquet — size: 47.65 MB


In [4]:
# Bronze principle: read as-is, no type casting, no filtering

df_source = spark.read.parquet(f"file://{local_file}")

print(f"Record count: {df_source.count():,}")
print("\nSchema:")
df_source.printSchema()
df_source.show(5, truncate=False)

Record count: 2,964,624

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+---------

In [5]:
# Add Bronze metadata columns

from pyspark.sql.functions import current_timestamp, lit, to_date

df_bronze = (
    df_source
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_ingestion_date",      to_date(current_timestamp()))
    .withColumn("_source_file",         lit(SOURCE_FILENAME))
    .withColumn("_ingestion_month",     lit(INGESTION_MONTH))
)

print(f"Bronze column count: {len(df_bronze.columns)}")
df_bronze.select(
    "_ingestion_timestamp", "_ingestion_date", "_source_file", "_ingestion_month"
).show(2, truncate=False)

Bronze column count: 23
+--------------------------+---------------+-------------------------------+----------------+
|_ingestion_timestamp      |_ingestion_date|_source_file                   |_ingestion_month|
+--------------------------+---------------+-------------------------------+----------------+
|2026-05-07 12:31:49.383194|2026-05-07     |yellow_tripdata_2024-01.parquet|2024-01         |
|2026-05-07 12:31:49.383194|2026-05-07     |yellow_tripdata_2024-01.parquet|2024-01         |
+--------------------------+---------------+-------------------------------+----------------+
only showing top 2 rows



In [6]:
# Partitioned by month for incremental load support

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ingestion_month")
    .save(BRONZE_YELLOW_PATH)
)

print(f"✓ Bronze write complete: {BRONZE_YELLOW_PATH}")

✓ Bronze write complete: /workspace/output/final_project/bronze/yellow


In [7]:
# Round-trip validation: read back from Delta, confirm row count matches source

df_bronze_check = spark.read.format("delta").load(BRONZE_YELLOW_PATH)

source_count = df_source.count()
bronze_count = df_bronze_check.count()

print(f"Source count : {source_count:,}")
print(f"Bronze count : {bronze_count:,}")
assert source_count == bronze_count, "Row count mismatch between source and Bronze"
print("✓ Row counts match")

print("\nDelta history:")
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_YELLOW_PATH}`").show(truncate=False)

Source count : 2,964,624
Bronze count : 2,964,624
✓ Row counts match

Delta history:
+-------+-----------------------+---------------+-------------+---------+------------------------------------------------------------------------------+----+------------------+--------------------+-----------+-----------------+-------------+---------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp              |userId         |userName     |operation|operationParameters                                                           |job |notebook          |clusterId           |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                     |userMetadata|engineInfo                                |
+-------+-----------------------+---------------+-------------+---------+------------------------------------------------------------------------------+----+------------------+---